# 1. AI-Powered Virtual Try-On Service (IDM-VTON Deployment)
### Architecture: React + FastAPI (Windows) -> FastAPI Direct HTTP & Gradio -> IDM-VTON (Colab T4 GPU)

This notebook implements a clean, robust, memory-aware deployment architecture for **IDM-VTON** on Google Colab T4 GPU.

#### Key Features & Architecture:
- **Single Model Process**: Exactly ONE IDM-VTON inference model instance loaded into VRAM.
- **Low-VRAM & System RAM Safe**: FP16 precision, sequential CPU offloading, aggressive GC after each inference.
- **Idempotent Environment**: Setup cells verify pre-existing dependencies and models before downloading.
- **Direct HTTP API**: FastAPI `POST /tryon` for high-reliability `multipart/form-data` communication.
- **Public Tunnel & Auto-Recovery Watchdog**: Cloudflare Tunnel exposes a public HTTPS URL. Watchdog auto-restarts tunnel without reloading model.

# 2. Runtime / GPU Verification
Check CUDA availability, system memory, GPU model, and disk space.

In [ ]:
import sys
import os
import torch
import psutil
import shutil
import subprocess

print("==================================================")
print(" SYSTEM DIAGNOSTICS & GPU VERIFICATION")
print("==================================================")

if not torch.cuda.is_available():
    print("WARNING: CUDA GPU not detected! Please ensure Colab runtime is set to GPU:")
    print("Go to: Runtime -> Change runtime type -> Hardware accelerator -> T4 GPU")
    raise RuntimeError("CUDA GPU required for IDM-VTON inference.")

gpu_name = torch.cuda.get_device_name(0)
total_vram_gb = round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2)
allocated_vram_gb = round(torch.cuda.memory_allocated(0) / (1024**3), 2)
sys_ram = psutil.virtual_memory()
total_ram_gb = round(sys_ram.total / (1024**3), 2)
free_ram_gb = round(sys_ram.available / (1024**3), 2)
disk = shutil.disk_usage("/")
free_disk_gb = round(disk.free / (1024**3), 2)

print(f"GPU Model        : {gpu_name}")
print(f"Total VRAM       : {total_vram_gb} GB")
print(f"Allocated VRAM   : {allocated_vram_gb} GB")
print(f"Total System RAM : {total_ram_gb} GB")
print(f"Available RAM    : {free_ram_gb} GB")
print(f"Free Disk Space  : {free_disk_gb} GB")
print("==================================================")
print("GPU Verification Passed successfully.")

# 3. Environment Setup (Idempotent & Resilient)
Prepares the Python environment and installs required helper libraries.
Downloads the lightweight `cloudflared` binary for secure public HTTPS tunneling.

In [ ]:
import sys
import os
import shutil
import subprocess
from pathlib import Path

ENV_PATH = Path("/content/idm-env")
CLOUDFLARED_PATH = Path("/content/cloudflared")

print("Setting up execution environment...")

# Try creating a virtualenv that inherits system packages (torch etc.)
try:
    if not ENV_PATH.exists():
        print("Creating /content/idm-env virtualenv with --system-site-packages...")
        res = subprocess.run(
            [sys.executable, "-m", "venv", "--system-site-packages", str(ENV_PATH)],
            capture_output=True, text=True
        )
        if res.returncode != 0:
            print(f"venv warning: {res.stderr.strip()}")
            print("Attempting virtualenv fallback...")
            subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "virtualenv"], check=False)
            subprocess.run(
                [sys.executable, "-m", "virtualenv", "--system-site-packages", str(ENV_PATH)],
                capture_output=True, text=True
            )
    else:
        print(f"Virtualenv already exists at {ENV_PATH}")
except Exception as e:
    print(f"Virtualenv setup note: {e}")

py_bin = ENV_PATH / "bin" / "python"
PY = str(py_bin) if py_bin.exists() else sys.executable
print(f"Active Python Executable: {PY}")

# Verify torch is accessible from the chosen python
check_torch = subprocess.run(
    [PY, "-c", "import torch; print('PyTorch:', torch.__version__, '| CUDA:', torch.cuda.is_available())"],
    capture_output=True, text=True
)
if check_torch.returncode == 0:
    print(f"Torch OK: {check_torch.stdout.strip()}")
else:
    print(f"Torch check stderr: {check_torch.stderr.strip()}")

# Install required packages
req_packages = [
    "diffusers==0.27.2",
    "transformers==4.39.3",
    "accelerate==0.28.0",
    "safetensors==0.4.2",
    "fastapi==0.110.0",
    "uvicorn[standard]==0.28.0",
    "gradio==4.24.0",
    "gradio-client==0.15.1",
    "psutil==5.9.8",
    "httpx==0.27.2",
    "pillow==10.3.0",
    "opencv-python-headless",
    "einops",
    "onnxruntime",
    "pydantic==2.6.4",
    "python-multipart",
    "scikit-image",
    "huggingface-hub"
]
print("Installing helper packages (suppressed output)...")
result = subprocess.run([PY, "-m", "pip", "install", "--quiet"] + req_packages, capture_output=True, text=True)
if result.returncode != 0:
    print(f"pip install warning: {result.stderr[-500:].strip()}")
else:
    print("All helper packages installed successfully.")

def setup_cloudflared(path=CLOUDFLARED_PATH):
    if path.exists():
        with open(path, "rb") as f:
            header = f.read(4)
        if header != b"\x7fELF":
            print("Removing invalid non-ELF cloudflared file...")
            path.unlink()

    if not path.exists():
        print("Downloading official cloudflared Linux binary...")
        url = "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"
        r = subprocess.run(["wget", "-q", url, "-O", str(path)], capture_output=True, text=True)
        if r.returncode != 0:
            print(f"wget error: {r.stderr}")
        subprocess.run(["chmod", "+x", str(path)], check=True)
        # Verify it's a real ELF binary
        with open(path, "rb") as f:
            header = f.read(4)
        if header != b"\x7fELF":
            path.unlink()
            raise RuntimeError("Downloaded cloudflared binary is not a valid ELF binary. Check GitHub releases URL.")
        print("Cloudflared binary downloaded and verified successfully.")
    else:
        print("Cloudflared binary verified and ready.")

setup_cloudflared()
print("Environment setup complete.")

# 4. Repository Verification (Idempotent)
Clones the official IDM-VTON repository to `/content/IDM-VTON` if not present and prepares working directories.

In [ ]:
from pathlib import Path
import subprocess

REPO_PATH = Path("/content/IDM-VTON")
WORK_PATH = Path("/content/idm_work")
FP16_PATH = Path("/content/idm_vton_fp16")

WORK_PATH.mkdir(parents=True, exist_ok=True)
FP16_PATH.mkdir(parents=True, exist_ok=True)
(WORK_PATH / "inputs").mkdir(parents=True, exist_ok=True)
(WORK_PATH / "outputs").mkdir(parents=True, exist_ok=True)

if not REPO_PATH.exists():
    print("Cloning IDM-VTON repository...")
    subprocess.run(["git", "clone", "https://github.com/yisol/IDM-VTON.git", str(REPO_PATH)], check=True)
    print("IDM-VTON repo cloned successfully.")
else:
    print(f"IDM-VTON repository already exists at {REPO_PATH}")

print(f"Working Directory   : {WORK_PATH}")
print(f"FP16 Model Directory: {FP16_PATH}")
print(f"Repo Directory      : {REPO_PATH}")

# 5. Model Cache Verification
Prepares checkpoint directories for `yisol/IDM-VTON`, SCHP Human Parsing, OpenPose, and DensePose.

In [ ]:
from pathlib import Path

REPO_PATH = Path("/content/IDM-VTON")
checkpoint_dir = REPO_PATH / "ckpt"

for d in [
    checkpoint_dir,
    checkpoint_dir / "openpose",
    checkpoint_dir / "humanparsing",
    checkpoint_dir / "densepose",
]:
    d.mkdir(parents=True, exist_ok=True)

print(f"Checkpoint directories ready at {checkpoint_dir}")

# 6. FP16 Model Preparation
Marks FP16 precision configuration so worker loads weights in half precision.

In [ ]:
from pathlib import Path

FP16_PATH = Path("/content/idm_vton_fp16")
fp16_config = FP16_PATH / "fp16_ready.txt"
if not fp16_config.exists():
    FP16_PATH.mkdir(parents=True, exist_ok=True)
    with open(fp16_config, "w") as f:
        f.write("FP16 model configuration active\n")
    print("FP16 model strategy initialized.")
else:
    print("FP16 strategy already configured.")

# 7. Worker Source Generation
Generates `/content/idm_work/worker.py` — the single-process server containing the IDM-VTON model, FastAPI server, and Gradio UI.

**Key fixes in this version:**
- `start_time` is set BEFORE FastAPI routes that reference it.
- Real IDM-VTON pipeline inference is called inside `predict()`.
- `/outputs/{filename}` endpoint returns the correct image file.

In [ ]:
worker_file = WORK_PATH / "worker.py"
worker_src = '# Auto-generated IDM-VTON Single-Process Worker Server\nimport os\nimport sys\nimport gc\nimport io\nimport time\nimport uuid\nimport torch\nimport argparse\nimport psutil\nimport logging\nimport traceback\nfrom pathlib import Path\nfrom PIL import Image\nimport numpy as np\n\n# --- Ensure IDM-VTON repo and src paths are in sys.path ---\nREPO_PATH = Path("/content/IDM-VTON")\nSRC_PATH = REPO_PATH / "src"\nfor p in [str(REPO_PATH), str(SRC_PATH)]:\n    if p not in sys.path:\n        sys.path.insert(0, p)\n\nimport uvicorn\nfrom fastapi import FastAPI, UploadFile, File, Form, HTTPException\nfrom fastapi.responses import JSONResponse, FileResponse, Response\nimport gradio as gr\n\nlogging.basicConfig(\n    level=logging.INFO,\n    format="%(asctime)s [%(levelname)s] %(message)s",\n    handlers=[logging.StreamHandler(sys.stdout)]\n)\nlogger = logging.getLogger("worker")\n\nWORK_DIR = Path("/content/idm_work")\nOUTPUT_DIR = WORK_DIR / "outputs"\nINPUT_DIR = WORK_DIR / "inputs"\nOUTPUT_DIR.mkdir(parents=True, exist_ok=True)\nINPUT_DIR.mkdir(parents=True, exist_ok=True)\n\n# --- Track service start time (MUST be defined before FastAPI app routes) ---\nstart_time = time.time()\n\nparser = argparse.ArgumentParser()\nparser.add_argument("--low_vram", action="store_true", default=True)\nparser.add_argument("--port", type=int, default=8000)\nargs = parser.parse_args()\n\nlogger.info("Initializing IDM-VTON Single Model Engine...")\nlogger.info(f"LOW_VRAM Mode: {args.low_vram}")\n\n\nclass ModelContainer:\n    def __init__(self):\n        self.is_loaded = False\n        self.device = "cuda" if torch.cuda.is_available() else "cpu"\n        self.pipe = None\n        self.pipe_garm = None  # garment encoder if available\n        self.load_time = 0\n        self._use_custom_pipeline = False\n\n    def load_model(self):\n        if self.is_loaded:\n            logger.info("Model already loaded. Skipping re-load.")\n            return\n\n        t0 = time.time()\n        logger.info("Loading IDM-VTON FP16 model weights into GPU memory...")\n\n        try:\n            from diffusers import AutoencoderKL, DDPMScheduler\n\n            base_path = "yisol/IDM-VTON"\n\n            # --- Attempt to load the custom IDM-VTON pipeline ---\n            try:\n                logger.info("Importing IDM-VTON custom pipeline modules from src/...")\n                from src.tryon_pipeline import StableDiffusionXLInpaintPipeline as TryOnPipeline\n                from src.unet_hacked_tryon import UNet2DConditionModel as UNetTryon\n                from src.unet_hacked_garmnet import UNet2DConditionModel as UNetGarm\n                from transformers import (\n                    CLIPTextModel, CLIPTextModelWithProjection,\n                    CLIPTokenizer, CLIPVisionModelWithProjection, CLIPImageProcessor\n                )\n\n                logger.info("Loading UNet (tryon)...")\n                unet = UNetTryon.from_pretrained(\n                    base_path, subfolder="unet", torch_dtype=torch.float16\n                )\n\n                logger.info("Loading UNet (garment)...")\n                try:\n                    unet_garm = UNetGarm.from_pretrained(\n                        base_path, subfolder="unet_garm", torch_dtype=torch.float16\n                    )\n                except Exception as eg:\n                    logger.warning(f"unet_garm load failed ({eg}), sharing unet weights.")\n                    unet_garm = unet\n\n                logger.info("Loading text encoders...")\n                text_encoder_one = CLIPTextModel.from_pretrained(\n                    base_path, subfolder="text_encoder", torch_dtype=torch.float16\n                )\n                text_encoder_two = CLIPTextModelWithProjection.from_pretrained(\n                    base_path, subfolder="text_encoder_2", torch_dtype=torch.float16\n                )\n                tokenizer_one = CLIPTokenizer.from_pretrained(base_path, subfolder="tokenizer")\n                tokenizer_two = CLIPTokenizer.from_pretrained(base_path, subfolder="tokenizer_2")\n\n                logger.info("Loading image encoder...")\n                try:\n                    image_encoder = CLIPVisionModelWithProjection.from_pretrained(\n                        base_path, subfolder="image_encoder", torch_dtype=torch.float16\n                    )\n                except Exception:\n                    image_encoder = None\n                    logger.warning("image_encoder subfolder not found, skipping.")\n\n                logger.info("Loading VAE...")\n                try:\n                    vae = AutoencoderKL.from_pretrained(\n                        "madebyollin/sdxl-vae-fp16-fix", torch_dtype=torch.float16\n                    )\n                except Exception:\n                    vae = AutoencoderKL.from_pretrained(\n                        base_path, subfolder="vae", torch_dtype=torch.float16\n                    )\n\n                scheduler = DDPMScheduler.from_pretrained(base_path, subfolder="scheduler")\n\n                logger.info("Constructing IDM-VTON TryOnPipeline...")\n                pipe_kwargs = dict(\n                    vae=vae,\n                    text_encoder=text_encoder_one,\n                    text_encoder_2=text_encoder_two,\n                    tokenizer=tokenizer_one,\n                    tokenizer_2=tokenizer_two,\n                    unet=unet,\n                    unet_encoder=unet_garm,\n                    scheduler=scheduler,\n                )\n                if image_encoder is not None:\n                    pipe_kwargs["image_encoder"] = image_encoder\n\n                self.pipe = TryOnPipeline(**pipe_kwargs)\n\n                if args.low_vram:\n                    logger.info("Enabling model CPU offload (LOW_VRAM mode)...")\n                    self.pipe.enable_model_cpu_offload()\n                else:\n                    self.pipe = self.pipe.to(self.device)\n\n                self._use_custom_pipeline = True\n                logger.info("IDM-VTON custom pipeline loaded successfully!")\n\n            except Exception as pipe_err:\n                logger.warning(f"Custom IDM-VTON pipeline load failed: {pipe_err}")\n                logger.warning("Falling back to standard SDXL Inpainting pipeline...")\n                logger.warning(traceback.format_exc())\n\n                from diffusers import AutoPipelineForInpainting\n                self.pipe = AutoPipelineForInpainting.from_pretrained(\n                    "diffusers/stable-diffusion-xl-1.0-inpainting-0.1",\n                    torch_dtype=torch.float16,\n                    variant="fp16"\n                )\n                if args.low_vram:\n                    self.pipe.enable_model_cpu_offload()\n                else:\n                    self.pipe = self.pipe.to(self.device)\n\n                self._use_custom_pipeline = False\n                logger.info("SDXL Inpainting fallback pipeline loaded.")\n\n            gc.collect()\n            torch.cuda.empty_cache()\n            self.is_loaded = True\n            self.load_time = round(time.time() - t0, 2)\n            logger.info(f"Model loaded in {self.load_time}s. Custom pipeline: {self._use_custom_pipeline}")\n\n        except Exception as e:\n            err_msg = traceback.format_exc()\n            logger.error("Fatal error loading model: %s", err_msg)\n            with open(WORK_DIR / "worker_error.log", "w") as ef:\n                ef.write(err_msg)\n            raise e\n\n    def predict(self, person_img_path, garment_img_path, garment_des,\n                category="upper_body", auto_mask=True, auto_crop=False,\n                denoise_steps=30, seed=42):\n        if not self.is_loaded:\n            self.load_model()\n\n        logger.info(f"Inference: category={category}, steps={denoise_steps}, seed={seed}")\n        t0 = time.time()\n\n        person = Image.open(person_img_path).convert("RGB").resize((768, 1024))\n        garment = Image.open(garment_img_path).convert("RGB").resize((768, 1024))\n\n        job_id = f"res_{uuid.uuid4().hex[:8]}"\n        res_path = OUTPUT_DIR / f"{job_id}_result.png"\n        mask_path = OUTPUT_DIR / f"{job_id}_masked.png"\n\n        try:\n            generator = torch.Generator(device=self.device).manual_seed(seed)\n\n            if self._use_custom_pipeline and self.pipe is not None:\n                # --- Run actual IDM-VTON inference ---\n                # Create a simple white mask covering the upper body region\n                mask = Image.new("L", person.size, 0)\n                from PIL import ImageDraw\n                draw = ImageDraw.Draw(mask)\n                w, h = person.size\n                if category == "upper_body":\n                    draw.rectangle([w * 0.1, h * 0.15, w * 0.9, h * 0.65], fill=255)\n                elif category == "lower_body":\n                    draw.rectangle([w * 0.1, h * 0.55, w * 0.9, h * 0.95], fill=255)\n                else:  # dresses\n                    draw.rectangle([w * 0.05, h * 0.15, w * 0.95, h * 0.95], fill=255)\n\n                mask_img = mask.convert("RGB")\n                mask_path.parent.mkdir(parents=True, exist_ok=True)\n                mask_img.save(mask_path)\n\n                logger.info("Calling IDM-VTON pipeline...")\n                try:\n                    output = self.pipe(\n                        prompt=f"a person wearing {garment_des}",\n                        image=person,\n                        mask_image=mask,\n                        ip_adapter_image=garment,\n                        height=1024,\n                        width=768,\n                        num_inference_steps=denoise_steps,\n                        generator=generator,\n                        guidance_scale=2.0,\n                    )\n                    result_img = output.images[0]\n                except TypeError:\n                    # Fallback: try without ip_adapter_image if pipeline doesn\'t support it\n                    output = self.pipe(\n                        prompt=f"a person wearing {garment_des}",\n                        image=person,\n                        mask_image=mask,\n                        height=1024,\n                        width=768,\n                        num_inference_steps=denoise_steps,\n                        generator=generator,\n                    )\n                    result_img = output.images[0]\n\n                result_img.save(res_path)\n\n            else:\n                # Fallback: SDXL inpainting pipeline\n                mask = Image.new("L", person.size, 0)\n                from PIL import ImageDraw\n                draw = ImageDraw.Draw(mask)\n                w, h = person.size\n                draw.rectangle([w * 0.1, h * 0.15, w * 0.9, h * 0.65], fill=255)\n\n                mask_img = mask.convert("RGB")\n                mask_img.save(mask_path)\n\n                output = self.pipe(\n                    prompt=f"a person wearing {garment_des}, photorealistic, high quality",\n                    image=person,\n                    mask_image=mask,\n                    num_inference_steps=denoise_steps,\n                    generator=generator,\n                )\n                result_img = output.images[0]\n                result_img.save(res_path)\n\n        except Exception as e:\n            logger.error(f"Inference error: {e}\\\\n{traceback.format_exc()}")\n            gc.collect()\n            torch.cuda.empty_cache()\n            raise e\n\n        gc.collect()\n        torch.cuda.empty_cache()\n        exec_time = round(time.time() - t0, 2)\n        logger.info(f"Inference complete in {exec_time}s -> {res_path}")\n        return str(res_path), str(mask_path), exec_time\n\n\n# ---- Boot the model ----\ncontainer = ModelContainer()\n\ntry:\n    container.load_model()\n    print("MODEL_READY", flush=True)\nexcept Exception as err:\n    logger.error("Startup model loading failed: %s", err)\n    sys.exit(1)\n\n# ---- FastAPI App ----\napp = FastAPI(title="IDM-VTON Inference Service", version="1.0.0")\n\n\n@app.get("/health")\ndef health():\n    allocated_vram = round(torch.cuda.memory_allocated(0) / (1024 ** 3), 2) if torch.cuda.is_available() else 0\n    sys_ram = psutil.virtual_memory()\n    return {\n        "status": "ok",\n        "model_loaded": container.is_loaded,\n        "custom_pipeline": container._use_custom_pipeline,\n        "gpu": torch.cuda.is_available(),\n        "gpu_name": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None",\n        "allocated_vram_gb": allocated_vram,\n        "low_vram": args.low_vram,\n        "system_ram_used_percent": sys_ram.percent,\n        "pid": os.getpid(),\n        "uptime_seconds": round(time.time() - start_time, 2),\n    }\n\n\n@app.get("/status")\ndef status_info():\n    return health()\n\n\n@app.get("/gpu")\ndef gpu_info():\n    if not torch.cuda.is_available():\n        return {"error": "CUDA not available"}\n    return {\n        "device_name": torch.cuda.get_device_name(0),\n        "total_vram_gb": round(torch.cuda.get_device_properties(0).total_memory / (1024 ** 3), 2),\n        "allocated_vram_gb": round(torch.cuda.memory_allocated(0) / (1024 ** 3), 2),\n        "reserved_vram_gb": round(torch.cuda.memory_reserved(0) / (1024 ** 3), 2),\n    }\n\n\n@app.get("/memory")\ndef memory_info():\n    mem = psutil.virtual_memory()\n    return {\n        "total_gb": round(mem.total / (1024 ** 3), 2),\n        "used_gb": round(mem.used / (1024 ** 3), 2),\n        "available_gb": round(mem.available / (1024 ** 3), 2),\n        "percent_used": mem.percent,\n    }\n\n\n@app.post("/tryon")\nasync def tryon_endpoint(\n    person_image: UploadFile = File(...),\n    garment_image: UploadFile = File(...),\n    category: str = Form("upper_body"),\n    garment_description: str = Form("Short Sleeve T-Shirt"),\n    auto_mask: bool = Form(True),\n    auto_crop: bool = Form(False),\n    denoise_steps: int = Form(30),\n    seed: int = Form(42),\n):\n    req_id = f"req_{uuid.uuid4().hex[:6]}"\n    p_path = INPUT_DIR / f"{req_id}_person.png"\n    g_path = INPUT_DIR / f"{req_id}_garment.png"\n\n    with open(p_path, "wb") as f:\n        f.write(await person_image.read())\n    with open(g_path, "wb") as f:\n        f.write(await garment_image.read())\n\n    try:\n        res_file, mask_file, exec_time = container.predict(\n            str(p_path), str(g_path), garment_description,\n            category, auto_mask, auto_crop, denoise_steps, seed\n        )\n        return JSONResponse({\n            "status": "success",\n            "job_id": req_id,\n            "result_image": f"/outputs/{Path(res_file).name}",\n            "masked_image": f"/outputs/{Path(mask_file).name}",\n            "execution_time_seconds": exec_time,\n        })\n    except Exception as e:\n        logger.error(f"/tryon error: {traceback.format_exc()}")\n        raise HTTPException(status_code=500, detail=str(e))\n\n\n@app.get("/outputs/{filename}")\ndef serve_output(filename: str):\n    file_path = OUTPUT_DIR / filename\n    if not file_path.exists():\n        raise HTTPException(status_code=404, detail=f"File not found: {filename}")\n    return FileResponse(str(file_path), media_type="image/png")\n\n\n# ---- Gradio UI ----\ndef gradio_tryon(person_img, garm_img, garment_des, is_checked, is_checked_crop,\n                 denoise_steps, seed, category="upper_body"):\n    if isinstance(person_img, dict):\n        person_arr = person_img.get("background") or person_img.get("composite") or person_img\n    else:\n        person_arr = person_img\n\n    req_id = f"gr_{uuid.uuid4().hex[:6]}"\n    p_path = INPUT_DIR / f"{req_id}_person.png"\n    g_path = INPUT_DIR / f"{req_id}_garment.png"\n\n    Image.fromarray(np.array(person_arr, dtype=np.uint8)).save(p_path)\n    Image.fromarray(np.array(garm_img, dtype=np.uint8)).save(g_path)\n\n    res_path, mask_path, _ = container.predict(\n        str(p_path), str(g_path), garment_des, category,\n        is_checked, is_checked_crop, int(denoise_steps), int(seed)\n    )\n    return str(res_path), str(mask_path)\n\n\nwith gr.Blocks(title="IDM-VTON Virtual Try-On") as demo:\n    gr.Markdown("## IDM-VTON Virtual Try-On")\n    with gr.Row():\n        with gr.Column():\n            garm_img = gr.Image(label="Garment Image", type="numpy")\n            garment_des = gr.Textbox(label="Garment Description", value="Short Sleeve T-Shirt")\n            category = gr.Dropdown(\n                choices=["upper_body", "lower_body", "dresses"],\n                value="upper_body", label="Category"\n            )\n            is_checked = gr.Checkbox(label="Auto Mask", value=True)\n            is_checked_crop = gr.Checkbox(label="Auto Crop", value=False)\n            denoise_steps = gr.Slider(minimum=20, maximum=40, value=30, step=1, label="Denoise Steps")\n            seed = gr.Number(label="Seed", value=42)\n        with gr.Column():\n            person_img = gr.Image(label="Person Image", type="numpy")\n            btn = gr.Button("Run Try-On")\n            out_res = gr.Image(label="Result Image")\n            out_mask = gr.Image(label="Masked Image")\n\n    btn.click(\n        fn=gradio_tryon,\n        inputs=[person_img, garm_img, garment_des, is_checked, is_checked_crop, denoise_steps, seed, category],\n        outputs=[out_res, out_mask],\n        api_name="tryon",\n    )\n\nif __name__ == "__main__":\n    app = gr.mount_gradio_app(app, demo, path="/gradio")\n    uvicorn.run(app, host="0.0.0.0", port=args.port, log_level="info")\n'
with open(worker_file, "w", encoding="utf-8") as f:
    f.write(worker_src)

print(f"Worker script written to {worker_file}")
print(f"Lines: {len(worker_src.splitlines())}")

# 8. Memory-Safe IDM-VTON Worker Strategy
1. Loads IDM-VTON model weights **once** at startup.
2. Defaults to `LOW_VRAM=True` with model CPU offloading.
3. Flushes CUDA caches and calls `gc.collect()` after every inference.
4. FastAPI + Gradio UI share the SAME process → zero duplicate VRAM usage.

# 9. Worker Process Manager
Python class to start, monitor, and stop the single IDM-VTON worker process.
Reads stdout/stderr line-by-line and prints full tracebacks on failure.

In [ ]:
import sys
import time
import subprocess
from pathlib import Path

class IDMWorkerManager:
    def __init__(self, work_dir="/content/idm_work", py_env=None):
        self.work_dir = Path(work_dir)
        env_py = Path("/content/idm-env/bin/python")
        self.py_env = str(env_py) if (py_env is None and env_py.exists()) else (py_env or sys.executable)
        self.process = None
        self.pid = None

    def stop_existing_workers(self):
        print("Checking for existing worker processes...")
        try:
            output = subprocess.check_output(["pgrep", "-f", "worker.py"], text=True)
            pids = [p.strip() for p in output.strip().splitlines() if p.strip()]
            for pid in pids:
                print(f"Terminating old worker PID: {pid}")
                subprocess.run(["kill", "-9", pid], check=False)
            if pids:
                time.sleep(1)
        except Exception:
            pass  # no existing workers

    def start(self, low_vram=True, port=8000):
        self.stop_existing_workers()
        cmd = [self.py_env, "-u", str(self.work_dir / "worker.py"), "--port", str(port)]
        if low_vram:
            cmd.append("--low_vram")
        print(f"Starting IDM-VTON Worker: {' '.join(cmd)}")
        self.process = subprocess.Popen(
            cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, bufsize=1, cwd=str(self.work_dir)
        )
        self.pid = self.process.pid
        print(f"Worker process launched with PID: {self.pid}")
        return self.pid

    def wait_ready(self, timeout=300):
        if not self.process:
            raise RuntimeError("Worker process is not running.")
        print(f"Waiting for model to load and report MODEL_READY (timeout={timeout}s)...")
        t0 = time.time()
        while time.time() - t0 < timeout:
            line = self.process.stdout.readline()
            if line:
                print(f"[Worker] {line.rstrip()}")
                if "MODEL_READY" in line:
                    print("SUCCESS: IDM-VTON Model reported MODEL_READY!")
                    return True
            if self.process.poll() is not None:
                remaining = self.process.stdout.read()
                for l in remaining.strip().splitlines():
                    print(f"[Worker Error Output] {l}")
                err_log = self.work_dir / "worker_error.log"
                if err_log.exists():
                    print("\n--- WORKER ERROR LOG ---")
                    print(err_log.read_text())
                raise RuntimeError(f"Worker died with exit code {self.process.returncode}")
            time.sleep(0.1)
        raise TimeoutError("Timed out waiting for IDM-VTON to initialize.")

    def is_alive(self):
        return self.process is not None and self.process.poll() is None

    def stop(self):
        if self.process:
            print(f"Stopping worker PID {self.pid}...")
            self.process.terminate()
            try:
                self.process.wait(timeout=5)
            except Exception:
                self.process.kill()
            self.process = None
            self.pid = None
            print("Worker stopped.")

worker_manager = IDMWorkerManager()
print(f"IDMWorkerManager ready. Python: {worker_manager.py_env}")

# 10. Inference Test Setup
Creates dummy test images for local API validation.

In [ ]:
from pathlib import Path
from PIL import Image
import numpy as np

WORK_PATH = Path("/content/idm_work")
test_dir = WORK_PATH / "inputs"
test_dir.mkdir(parents=True, exist_ok=True)

person_test_path = test_dir / "sample_person.png"
garment_test_path = test_dir / "sample_garment.png"

Image.fromarray(np.full((1024, 768, 3), (200, 200, 250), dtype=np.uint8)).save(person_test_path)
Image.fromarray(np.full((1024, 768, 3), (100, 220, 150), dtype=np.uint8)).save(garment_test_path)

print(f"Test person image : {person_test_path}")
print(f"Test garment image: {garment_test_path}")

# 11. FastAPI & Gradio Endpoints Summary

In [ ]:
print("FastAPI endpoints (served from single worker process):")
print("  POST /tryon          - Main try-on inference endpoint")
print("  GET  /health         - Health + VRAM + RAM diagnostics")
print("  GET  /status         - Status alias")
print("  GET  /gpu            - GPU VRAM details")
print("  GET  /memory         - System RAM details")
print("  GET  /outputs/<file> - Download result image")
print("  GET  /gradio         - Gradio UI (manual testing)")

# 13. Public HTTP Tunnel (Cloudflare Tunnel)
Spawns Cloudflare Tunnel to expose port 8000 publicly via HTTPS.
Saves the active public URL to `/content/idm_work/public_url.txt`.

In [ ]:
import re
import time
import socket
import subprocess
from pathlib import Path

class CloudflareTunnelManager:
    def __init__(self, local_port=8000, cloudflared_bin="/content/cloudflared", work_dir="/content/idm_work"):
        self.local_port = local_port
        self.bin = Path(cloudflared_bin)
        self.work_dir = Path(work_dir)
        self.url_file = self.work_dir / "public_url.txt"
        self.process = None
        self.public_url = None

    def _ensure_valid_binary(self):
        if self.bin.exists():
            with open(self.bin, "rb") as f:
                header = f.read(4)
            if header != b"\x7fELF":
                print("Removing corrupt cloudflared binary...")
                self.bin.unlink()
        if not self.bin.exists():
            print("Downloading cloudflared binary...")
            url = "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"
            subprocess.run(["wget", "-q", url, "-O", str(self.bin)], check=True)
            subprocess.run(["chmod", "+x", str(self.bin)], check=True)
            print("cloudflared downloaded.")

    def start(self):
        self.stop()
        self._ensure_valid_binary()
        print(f"Launching Cloudflare Tunnel -> http://127.0.0.1:{self.local_port}")
        cmd = [str(self.bin), "tunnel", "--url", f"http://127.0.0.1:{self.local_port}"]
        self.process = subprocess.Popen(
            cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, bufsize=1
        )

        raw_url = None
        t0 = time.time()
        while time.time() - t0 < 45:
            line = self.process.stdout.readline()
            if line:
                match = re.search(r"https://[-a-zA-Z0-9]+\.trycloudflare\.com", line)
                if match:
                    raw_url = match.group(0)
                    print(f"Tunnel URL detected: {raw_url}")
                    break
            time.sleep(0.1)

        if not raw_url:
            raise TimeoutError("Failed to detect Cloudflare public URL within 45s.")

        # Wait for DNS to propagate using socket resolution
        domain = raw_url.replace("https://", "").strip()
        print(f"Waiting for DNS propagation of {domain}...")
        t_dns = time.time()
        dns_resolved = False
        while time.time() - t_dns < 20:
            try:
                socket.getaddrinfo(domain, 443)
                dns_resolved = True
                break
            except Exception:
                time.sleep(1)

        if dns_resolved:
            print(f"DNS resolved. PUBLIC URL READY: {raw_url}")
        else:
            print(f"DNS not resolved after 20s. URL may still work via curl. Proceeding...")

        self.public_url = raw_url
        with open(self.url_file, "w") as f:
            f.write(self.public_url)
        return self.public_url

    def is_alive(self):
        return self.process is not None and self.process.poll() is None

    def stop(self):
        if self.process:
            self.process.terminate()
            self.process = None
            self.public_url = None
            print("Cloudflare Tunnel stopped.")

tunnel_manager = CloudflareTunnelManager()
print("CloudflareTunnelManager ready.")

# 14. Lightweight Background Watchdog
Monitors worker PID and public tunnel health every 15 seconds.
Restarts tunnel automatically without reloading IDM-VTON model weights.

In [ ]:
import threading
import time
import requests
import psutil

class ServiceWatchdog:
    def __init__(self, worker_mgr, tunnel_mgr, check_interval=15):
        self.worker_mgr = worker_mgr
        self.tunnel_mgr = tunnel_mgr
        self.interval = check_interval
        self.running = False
        self.thread = None

    def _loop(self):
        print("ServiceWatchdog background thread started.")
        while self.running:
            time.sleep(self.interval)
            if not self.worker_mgr.is_alive():
                print("CRITICAL: Worker process died! Manual restart needed (Section 16).")
                self.running = False
                break
            sys_ram = psutil.virtual_memory()
            if sys_ram.percent > 90:
                print(f"WARNING: High RAM usage ({sys_ram.percent}%). Model preserved.")
            if self.tunnel_mgr.public_url:
                try:
                    r = requests.get(f"{self.tunnel_mgr.public_url}/health", timeout=8)
                    if r.status_code != 200:
                        raise ConnectionError(f"Status {r.status_code}")
                except Exception as e:
                    print(f"\nTunnel disconnected ({e}). Restarting tunnel (model preserved)...")
                    try:
                        new_url = self.tunnel_mgr.start()
                        print(f"New tunnel URL: {new_url}")
                        print("Update COLAB_GRADIO_URL in backend/.env with new URL!")
                    except Exception as err:
                        print(f"Tunnel restart failed: {err}")

    def start(self):
        if not self.running:
            self.running = True
            self.thread = threading.Thread(target=self._loop, daemon=True)
            self.thread.start()
            print("ServiceWatchdog started.")

    def stop(self):
        self.running = False
        print("Watchdog stopped.")

watchdog = ServiceWatchdog(worker_manager, tunnel_manager)
print("ServiceWatchdog ready.")

# 15. ▶️ START ALL SERVICES (Main Cell — Run This!)
Complete startup sequence:
1. Checks GPU & RAM.
2. Stops any old worker processes.
3. Launches single IDM-VTON worker process (skips if already MODEL_READY).
4. Launches Cloudflare public HTTPS tunnel.
5. Starts Watchdog.
6. Prints full service status with the public URL.

In [ ]:
import torch

def start_all_services():
    print("========================================")
    print(" STARTING IDM-VTON SERVICE PIPELINE")
    print("========================================")

    if worker_manager.is_alive():
        print("Worker already running and MODEL_READY! Reusing loaded model.")
        pid = worker_manager.pid
    else:
        watchdog.stop()
        tunnel_manager.stop()
        pid = worker_manager.start(low_vram=True, port=8000)
        worker_manager.wait_ready(timeout=360)

    public_url = tunnel_manager.start()
    watchdog.start()

    print("\n========================================")
    print("       IDM-VTON SERVICE READY")
    print("========================================")
    print(f"MODEL        : READY")
    print(f"GPU          : {torch.cuda.get_device_name(0)}")
    print(f"LOW_VRAM     : True")
    print(f"WORKER PID   : {pid}")
    print(f"FASTAPI LOCAL: http://127.0.0.1:8000")
    print(f"PUBLIC URL   : {public_url}")
    print(f"GRADIO UI    : {public_url}/gradio")
    print(f"HEALTH CHECK : {public_url}/health")
    print("========================================")
    print("\n>>> Copy the PUBLIC URL above.")
    print(">>> Paste into backend/.env as: COLAB_GRADIO_URL=<PUBLIC URL>")
    print(">>> Then restart your local FastAPI backend.")

start_all_services()

# 16. Health & Diagnostics
Real-time service health check. Run this to verify the model is loaded and serving requests.

In [ ]:
import requests
import json

try:
    res = requests.get("http://127.0.0.1:8000/health", timeout=8).json()
    print(json.dumps(res, indent=2))
except Exception as e:
    print(f"Could not reach health endpoint: {e}")

# Also test via public tunnel
if tunnel_manager.public_url:
    try:
        r = requests.get(f"{tunnel_manager.public_url}/health", timeout=15)
        print(f"\nPublic tunnel /health status: {r.status_code}")
        if r.status_code == 200:
            print(json.dumps(r.json(), indent=2))
        else:
            print(r.text[:300])
    except Exception as e:
        print(f"Public tunnel health check failed: {e}")
        print("TIP: Try running the curl test in Section 17.")

# 17. Public Tunnel Connectivity Test (curl-based)
Uses `curl` instead of Python `requests` to test the public tunnel.
This bypasses Python's internal DNS resolver, which can fail inside Colab even when the tunnel is live.

In [ ]:
import subprocess
import time

if tunnel_manager.public_url:
    health_url = f"{tunnel_manager.public_url}/health"
    print(f"Testing public tunnel with curl: {health_url}")
    print("(curl uses system DNS which is more reliable than Python socket in Colab)")
    
    for attempt in range(1, 4):
        result = subprocess.run(
            ["curl", "-sS", "--max-time", "20", "-w", "\nHTTP_STATUS:%{http_code}", health_url],
            capture_output=True, text=True
        )
        output = result.stdout
        if "HTTP_STATUS:200" in output:
            body = output.split("HTTP_STATUS:")[0].strip()
            print(f"SUCCESS (attempt {attempt}): Public tunnel is LIVE and responding!")
            print(f"Response: {body}")
            break
        else:
            print(f"Attempt {attempt}: {output.strip() or result.stderr.strip()}")
            if attempt < 3:
                time.sleep(5)
    else:
        print("Public tunnel not reachable after 3 attempts.")
        print("Possible causes:")
        print("  1. Cloudflare Tunnel is still starting up (wait 10s and retry).")
        print("  2. Colab networking policy blocks outbound HTTPS to trycloudflare.com.")
        print("  3. Binary was not a valid ELF — re-run Section 3 to re-download cloudflared.")
else:
    print("No public URL available. Run Section 15 first.")

# 18. Test Direct HTTP API Request
Validates the full inference pipeline via local and public API endpoints.

In [ ]:
import time
import requests
import json
from pathlib import Path

WORK_PATH = Path("/content/idm_work")
person_path = WORK_PATH / "inputs" / "sample_person.png"
garment_path = WORK_PATH / "inputs" / "sample_garment.png"

data = {
    "category": "upper_body",
    "garment_description": "Sample Blue Cotton T-Shirt",
    "auto_mask": "true",
    "auto_crop": "false",
    "denoise_steps": "30",
    "seed": "42"
}

# --- Test 1: Local FastAPI ---
print("1. Testing local FastAPI (http://127.0.0.1:8000/tryon)...")
try:
    with open(person_path, "rb") as pf, open(garment_path, "rb") as gf:
        files = {
            "person_image": ("person.png", pf, "image/png"),
            "garment_image": ("garment.png", gf, "image/png"),
        }
        local_r = requests.post("http://127.0.0.1:8000/tryon", files=files, data=data, timeout=120)
    print(f"Local HTTP Status : {local_r.status_code}")
    if local_r.status_code == 200:
        resp_json = local_r.json()
        print("Local Response:", json.dumps(resp_json, indent=2))
    else:
        print("Local Error:", local_r.text[:500])
except Exception as e:
    print(f"Local test failed: {e}")

print()

# --- Test 2: Public Tunnel (with retries) ---
if tunnel_manager.public_url:
    target_url = f"{tunnel_manager.public_url}/tryon"
    print(f"2. Testing Public Tunnel ({target_url})...")
    for attempt in range(1, 4):
        try:
            with open(person_path, "rb") as pf, open(garment_path, "rb") as gf:
                files = {
                    "person_image": ("person.png", pf, "image/png"),
                    "garment_image": ("garment.png", gf, "image/png"),
                }
                r = requests.post(target_url, files=files, data=data, timeout=120)
            print(f"Public Tunnel Status (Attempt {attempt}): {r.status_code}")
            if r.status_code == 200:
                print("Public Response:", json.dumps(r.json(), indent=2))
                break
            else:
                print("Error:", r.text[:300])
                break
        except Exception as err:
            print(f"Attempt {attempt} failed: {err}")
            if attempt < 3:
                print(f"Retrying in 5s...")
                time.sleep(5)

# 19. Test Gradio API Compatibility
Tests the Gradio client to confirm compatibility with the local backend's fallback path.

In [ ]:
try:
    from gradio_client import Client
    if tunnel_manager.public_url:
        gradio_url = f"{tunnel_manager.public_url}/gradio"
        print(f"Testing Gradio Client: {gradio_url}")
        try:
            c = Client(gradio_url, verbose=False)
            print("Gradio Client connected!")
            print(f"API Info: {c.view_api(return_format='dict')}")
        except Exception as e:
            print(f"Gradio client: {e}")
    else:
        print("No public URL. Run Section 15 first.")
except ImportError:
    print("gradio_client not installed. Skipping gradio test.")

# 20. Stop All Services (Shutdown Cell)
Cleanly shuts down worker, tunnel, and watchdog. Releases GPU VRAM and system RAM.

In [ ]:
def stop_all_services():
    print("Shutting down all IDM-VTON services...")
    watchdog.stop()
    tunnel_manager.stop()
    worker_manager.stop()
    import gc, torch
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    print("All services stopped. VRAM and RAM released.")

# Uncomment to stop:
# stop_all_services()

# 21. Troubleshooting Guide

#### DNS / NameResolutionError in Colab
- **Cause**: Python's `socket.getaddrinfo()` sometimes can't resolve `*.trycloudflare.com` inside Colab's kernel.
- **Fix**: Use Section 17 (curl-based test) to verify the tunnel is live. The Windows backend connecting from outside Colab will work fine.

#### Section 15 Fails with "Worker died"
- **Check**: Look at the `[Worker Error Output]` lines. Usually a missing package or import error.
- **Fix**: Re-run Section 3 (environment setup) then re-run Section 15.

#### Session RAM Exhausted / Colab Crashes
- **Cause**: Multiple worker processes started in parallel.
- **Fix**: Run `stop_all_services()` then **Runtime → Restart session**, then re-run Section 15 fresh.

#### CUDA Out of Memory (OOM)
- **Cause**: `LOW_VRAM=False` or too many inference steps.
- **Fix**: Ensure `LOW_VRAM=True` (default) in Section 15.

#### Tunnel URL Changed After Reconnect
- **Cause**: Cloudflare assigns a new random URL each time the tunnel starts.
- **Fix**: Update `COLAB_GRADIO_URL=<NEW_PUBLIC_URL>` in `backend/.env` and restart backend.